In [3]:
import io
import os  # Import os module for path manipulation
from pathlib import Path

import fitz  # Import fitz from PyMuPDF
import pytesseract
from PIL import Image, ImageFilter, ImageOps

# Use local file upload mechanism since running on your PC
class _LocalFiles:
    @staticmethod
    def upload():
        print("Enter local file paths separated by commas:")
        raw_paths = input().strip()

        uploaded_files = {}
        for raw_path in raw_paths.split(","):
            file_path = Path(raw_path.strip()).expanduser()
            if file_path.is_file():
                uploaded_files[file_path.name] = file_path.read_bytes()
            else:
                print(f"Skipping missing file: {file_path}")

        if not uploaded_files:
            print("No valid files were provided.")
        return uploaded_files

files = _LocalFiles()


ModuleNotFoundError: No module named 'frontend'

In [2]:
uploaded = files.upload()

Enter local file paths separated by commas:


In [ ]:
print("Extracting text from uploaded files...\n")

zoom = 3  # Increase for higher OCR resolution
mat = fitz.Matrix(zoom, zoom)
tesseract_lang = "eng"  # Change if your document is non-English


def preprocess_image(image: Image.Image) -> Image.Image:
    """Gentle cleanup to help OCR — avoids destroying text detail."""
    # Convert to grayscale
    gray = image.convert("L")
    # Light denoise — size=3 is fine for most scans
    denoised = gray.filter(ImageFilter.MedianFilter(size=3))
    # Stretch contrast so dim text becomes darker
    boosted = ImageOps.autocontrast(denoised, cutoff=0.5)
    # Return as RGB for Tesseract (skip aggressive binary threshold)
    return boosted.convert("RGB")


def ocr_image_bytes(raw_bytes: bytes) -> str:
    """Run Tesseract OCR on image bytes."""
    image = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
    cleaned = preprocess_image(image)
    # PSM 3 = fully automatic page segmentation (handles columns, mixed layouts)
    return pytesseract.image_to_string(
        cleaned, lang=tesseract_lang, config="--oem 3 --psm 3"
    )


def ocr_pdf_document(pdf_document: fitz.Document) -> str:
    """Render each PDF page to an image and OCR it."""
    ocr_pages = []
    for page_num, page in enumerate(pdf_document, start=1):
        pix = page.get_pixmap(matrix=mat)
        image = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        cleaned = preprocess_image(image)
        page_text = pytesseract.image_to_string(
            cleaned, lang=tesseract_lang, config="--oem 3 --psm 3"
        ).strip()
        ocr_pages.append(f"--- Page {page_num} ---\n{page_text}")
    return "\n\n".join(ocr_pages).strip()


def extract_pdf_text(raw_bytes: bytes) -> str:
    """Extract text from PDF; fallback to OCR for scanned/image-based PDFs."""
    pdf_document = fitz.open(stream=raw_bytes, filetype="pdf")
    try:
        native_pages = []
        has_images = False

        for page in pdf_document:
            page_text = page.get_text().strip()
            if page_text:
                native_pages.append(page_text)
            # Check if the page contains images (sign of a scanned PDF)
            if page.get_images():
                has_images = True

        combined_native = "\n\n".join(native_pages).strip()

        # If we got meaningful native text, return it
        if combined_native and len(combined_native) > 50:
            return combined_native

        # If page has images but little/no text → image-based/scanned PDF
        if has_images or not combined_native:
            print("Scanned / image-based PDF detected; running OCR on each page...")
            ocr_text = ocr_pdf_document(pdf_document)
            return ocr_text

        return combined_native
    finally:
        pdf_document.close()


# Bail out early if nothing was uploaded
if not uploaded:
    print("No files to process.")
else:
    file_names = list(uploaded.keys())
    num_files = len(file_names)

    for index, file_name in enumerate(file_names):
        print(f"Processing file: {file_name}")

        file_extension = os.path.splitext(file_name)[1].lower()

        if file_extension in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]:
            print(f"------ Text from Image: {file_name} ------")
            try:
                text = ocr_image_bytes(uploaded[file_name])
                print(text or "[No OCR text returned]")
            except Exception as e:
                print(f"Error processing image {file_name}: {e}")
        elif file_extension == ".pdf":
            print(f"------ Text from PDF: {file_name} ------")
            try:
                pdf_text = extract_pdf_text(uploaded[file_name])
                print(pdf_text or "[No OCR text returned]")
            except Exception as e:
                print(f"Error processing PDF {file_name}: {e}")
        else:
            print(f"Skipping non-image/non-PDF file: {file_name}")

        if num_files > 1 and index < num_files - 1:
            print("\n================ New File Text ===================\n")

Extracting text from uploaded files...

Processing file: WhatsApp Image 2026-02-26 at 2.27.12 PM.jpeg
------ Text from Image: WhatsApp Image 2026-02-26 at 2.27.12 PM.jpeg ------
qT ey ate | _ Se ieiind » —" ume aey
‘be - F a
a if ——— Hi Th ona war 8 TR tay OR ‘ ae ,
= 1 OBS fF ~o i te iF i ree - pardon: +
; 2, Ai “(eur HOW?” ! PY ah een toe a npantor,
vt | ee |i A k(t ee tuith mer
eee = My ’ tf Dee 1 ee C3 | peheving the unbelter ubie Bors
5 p ! N 3 “y , tneuns hoping when vv ervthung
%, | im
fa Z 1 =f — I seems hopeless
~ nS | Sra a hk (y= s(BA wee OK Chesterton
Begin By Showing Kindness To Yourself
(arapenl Ganesh ‘when I lakes bravery We havetobe To way you can lovoothers or recetve needs dedicated dally practize As:
in able to start from a place of radical hove. Regardless of who you are or saying goes. practice makes perfor:
ooftten Gas roles as parents. acceptance. Learning to work with what you do. it isagtven thateveryune Whether one is resolving to becomef:
children, ings. cou